# DS2/GSE141383 — build the raw-count AnnData object

This notebook reconstructs DS2 directly from the nine GEO supplementary
count tables extracted from `GSE141383_RAW.tar`.

Each file is a gzip-compressed, tab-delimited gene-by-cell matrix:

- column 1: gene identifier;
- column 2: gene symbol;
- remaining columns: raw molecular counts for individual cells;
- header: cell-identifying barcode for every cell column.

The wide matrices are read in row chunks to limit memory use. Each sample
is converted to a sparse cell-by-gene AnnData object and all nine samples
are concatenated after validating their feature axes.

## Expected result

- 9 samples
- 29,415 cells
- 60,725 features

## Default paths

Input directory:

`data/raw/GSE141383/`

Output:

`data/interim/DS2_GSE141383/GSE141383_merged.h5ad`

Override paths with:

- `GLIOMA_PROJECT_ROOT`
- `GSE141383_RAW_DIR`
- `GSE141383_MERGED_H5AD`
- `GSE141383_CHUNK_ROWS`

In [1]:
import gc
import gzip
import json
import os
import re
import sys
from datetime import datetime
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

print("Python:", sys.version)
print("anndata:", ad.__version__)
print("scanpy:", sc.__version__)
print("pandas:", pd.__version__)

Python: 3.12.3 | packaged by conda-forge | (main, Apr 15 2024, 18:20:11) [MSC v.1938 64 bit (AMD64)]
anndata: 0.12.7
scanpy: 1.11.5
pandas: 2.3.3


C:\Users\AmirSilco\AppData\Local\Temp\ipykernel_12364\209130806.py:17: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print("anndata:", ad.__version__)
C:\Users\AmirSilco\AppData\Local\Temp\ipykernel_12364\209130806.py:18: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("scanpy:", sc.__version__)


In [2]:
def find_repository_root(start: Path) -> Path:
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start


PROJECT_ROOT = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", find_repository_root(Path.cwd()))
).expanduser().resolve()

RAW_DIR = Path(
    os.environ.get(
        "GSE141383_RAW_DIR",
        PROJECT_ROOT / "data" / "raw" / "GSE141383",
    )
).expanduser().resolve()

OUTPUT_H5AD = Path(
    os.environ.get(
        "GSE141383_MERGED_H5AD",
        PROJECT_ROOT
        / "data"
        / "interim"
        / "DS2_GSE141383"
        / "GSE141383_merged.h5ad",
    )
).expanduser().resolve()

AUDIT_DIR = (
    PROJECT_ROOT / "results" / "qc" / "DS2_GSE141383" / "00a_build"
)

CHUNK_ROWS = int(os.environ.get("GSE141383_CHUNK_ROWS", "250"))

EXPECTED_FILES = {
    "GSM4202144": "PJ052",
    "GSM4202145": "PJ053",
    "GSM4202146": "PW016-703",
    "GSM4202147": "PW017-703",
    "GSM4202148": "PW032-706",
    "GSM4202149": "PW032-710",
    "GSM5975476": "PW035-710",
    "GSM5975477": "PDC001",
    "GSM5975478": "PW039-705",
}

EXPECTED_TOTAL_CELLS = 29_415
EXPECTED_N_FEATURES = 60_725

OUTPUT_H5AD.parent.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("RAW directory:", RAW_DIR)
print("Output:", OUTPUT_H5AD)
print("Chunk rows:", CHUNK_ROWS)

if not RAW_DIR.exists():
    raise FileNotFoundError(
        f"RAW count-table directory not found: {RAW_DIR}\n"
        "Extract the nine *.counts.txt.gz files there or set "
        "GSE141383_RAW_DIR."
    )

Project root: C:\Users\AmirSilco\Documents\GitHub\glioma-cnv-single-cell-atlas
RAW directory: C:\Users\AmirSilco\Documents\GitHub\glioma-cnv-single-cell-atlas\data\raw\GSE141383
Output: C:\Users\AmirSilco\Documents\GitHub\glioma-cnv-single-cell-atlas\data\interim\DS2_GSE141383\GSE141383_merged.h5ad
Chunk rows: 250


In [3]:
FILE_PATTERN = re.compile(
    r"^(GSM\d+)_(.+)\.counts\.txt\.gz$"
)


def discover_input_files(raw_dir: Path) -> list[dict]:
    records = []

    for path in sorted(raw_dir.glob("GSM*.counts.txt.gz")):
        match = FILE_PATTERN.match(path.name)
        if match is None:
            continue

        geo_sample_id, sample_id = match.groups()
        records.append(
            {
                "path": path,
                "geo_sample_id": geo_sample_id,
                "sample_id": sample_id,
            }
        )

    observed = {
        record["geo_sample_id"]: record["sample_id"]
        for record in records
    }

    if observed != EXPECTED_FILES:
        missing = sorted(set(EXPECTED_FILES) - set(observed))
        extra = sorted(set(observed) - set(EXPECTED_FILES))
        mismatched = {
            key: (EXPECTED_FILES.get(key), observed.get(key))
            for key in set(EXPECTED_FILES) & set(observed)
            if EXPECTED_FILES[key] != observed[key]
        }
        raise ValueError(
            "The discovered files do not match the expected nine-sample "
            f"GSE141383 set. Missing={missing}; extra={extra}; "
            f"mismatched={mismatched}"
        )

    return records


input_records = discover_input_files(RAW_DIR)

file_inventory = pd.DataFrame(
    [
        {
            "geo_sample_id": record["geo_sample_id"],
            "sample_id": record["sample_id"],
            "filename": record["path"].name,
            "compressed_size_bytes": record["path"].stat().st_size,
        }
        for record in input_records
    ]
)
file_inventory.to_csv(
    AUDIT_DIR / "GSE141383_input_file_inventory.tsv",
    sep="\t",
    index=False,
)

file_inventory

,geo_sample_id,sample_id,filename,compressed_size_bytes
0,GSM4202144,PJ052,GSM4202144_PJ052.counts.txt.gz,3962554
1,GSM4202145,PJ053,GSM4202145_PJ053.counts.txt.gz,7448304
2,GSM4202146,PW016-703,GSM4202146_PW016-703.counts.txt.gz,4620553
3,GSM4202147,PW017-703,GSM4202147_PW017-703.counts.txt.gz,4209963
4,GSM4202148,PW032-706,GSM4202148_PW032-706.counts.txt.gz,9185488
5,GSM4202149,PW032-710,GSM4202149_PW032-710.counts.txt.gz,2860709
6,GSM5975476,PW035-710,GSM5975476_PW035-710.counts.txt.gz,3064946
7,GSM5975477,PDC001,GSM5975477_PDC001.counts.txt.gz,13233219
8,GSM5975478,PW039-705,GSM5975478_PW039-705.counts.txt.gz,3325181


In [4]:
def inspect_header(path: Path) -> tuple[str, str, list[str]]:
    with gzip.open(path, "rt", encoding="utf-8", newline="") as handle:
        header_line = handle.readline()

    if not header_line:
        raise ValueError(f"{path.name} is empty.")

    fields = header_line.rstrip("\r\n").split("\t")
    if len(fields) < 3:
        raise ValueError(
            f"{path.name}: expected two gene columns and at least one "
            f"cell column, observed {len(fields)} columns."
        )

    gene_id_label, gene_symbol_label = fields[:2]
    cell_barcodes = fields[2:]

    if any(barcode == "" for barcode in cell_barcodes):
        raise ValueError(f"{path.name}: an empty cell barcode was found.")
    if len(cell_barcodes) != len(set(cell_barcodes)):
        raise ValueError(
            f"{path.name}: duplicate cell barcodes were found in the header."
        )

    return gene_id_label, gene_symbol_label, cell_barcodes


def numeric_count_array(frame: pd.DataFrame, source: Path) -> np.ndarray:
    try:
        numeric = frame.apply(pd.to_numeric, errors="raise")
    except Exception as exc:
        raise ValueError(
            f"A non-numeric count was found in {source.name}."
        ) from exc

    values = numeric.to_numpy(copy=False)

    if not np.isfinite(values).all():
        raise ValueError(f"Non-finite count found in {source.name}.")
    if np.any(values < 0):
        raise ValueError(f"Negative count found in {source.name}.")
    if not np.equal(values, np.floor(values)).all():
        raise ValueError(f"Non-integer count found in {source.name}.")

    maximum = values.max(initial=0)
    if maximum > np.iinfo(np.int32).max:
        raise OverflowError(
            f"A count in {source.name} exceeds int32: {maximum}"
        )

    return values.astype(np.int32, copy=False)


def read_geo_count_table(
    path: Path,
    geo_sample_id: str,
    sample_id: str,
    chunk_rows: int,
) -> ad.AnnData:
    (
        gene_id_header,
        gene_symbol_header,
        cell_barcodes,
    ) = inspect_header(path)

    expected_columns = [gene_id_header, gene_symbol_header, *cell_barcodes]

    gene_ids = []
    gene_symbols = []
    matrix_blocks = []

    reader = pd.read_csv(
        path,
        sep="\t",
        compression="gzip",
        header=0,
        chunksize=chunk_rows,
        low_memory=False,
    )

    for chunk_number, chunk in enumerate(reader, start=1):
        if chunk.columns.tolist() != expected_columns:
            raise ValueError(
                f"{path.name}, chunk {chunk_number}: parsed column names "
                "do not match the file header."
            )

        gene_ids.extend(chunk.iloc[:, 0].astype(str).tolist())
        gene_symbols.extend(chunk.iloc[:, 1].astype(str).tolist())

        counts = numeric_count_array(chunk.iloc[:, 2:], path)
        matrix_blocks.append(sp.csr_matrix(counts.T))

        del chunk, counts
        gc.collect()

    if not matrix_blocks:
        raise ValueError(f"No data rows were read from {path.name}.")

    X = sp.hstack(matrix_blocks, format="csr").astype(
        np.int32,
        copy=False,
    )
    del matrix_blocks
    gc.collect()

    feature_index = pd.Index(gene_ids, name="gene_id")
    if not feature_index.is_unique:
        duplicate_examples = (
            feature_index[feature_index.duplicated(keep=False)]
            .unique()
            .tolist()[:10]
        )
        raise ValueError(
            f"{path.name}: duplicate gene identifiers were found, "
            f"including {duplicate_examples}"
        )

    original_barcodes = pd.Index(
        [str(barcode) for barcode in cell_barcodes],
        name="cell_barcode",
    )
    cell_ids = pd.Index(
        [f"{sample_id}:{barcode}" for barcode in original_barcodes],
        name="cell_id",
    )

    obs = pd.DataFrame(index=cell_ids)
    obs["cell_barcode"] = original_barcodes.to_numpy()
    obs["sample"] = sample_id
    obs["sample_id"] = sample_id
    obs["batch"] = sample_id
    obs["geo_sample_id"] = geo_sample_id
    obs["geo_id"] = "GSE141383"
    obs["core_dataset"] = "DS2_GSE141383"
    obs["source_file"] = path.name

    var = pd.DataFrame(
        {
            "gene_symbols": pd.Series(
                gene_symbols,
                dtype="string",
            ).fillna("").astype(str).to_numpy(),
        },
        index=feature_index,
    )

    sample = ad.AnnData(X=X, obs=obs, var=var)
    sample.uns["source_file"] = path.name
    sample.uns["source_geo_accession"] = geo_sample_id
    sample.uns["source_gene_id_header"] = gene_id_header
    sample.uns["source_gene_symbol_header"] = gene_symbol_header

    return sample

In [5]:
sample_objects = []
sample_summary_rows = []
reference_var = None
reference_features = None

for record in input_records:
    print(f"Reading {record['path'].name} ...")

    sample = read_geo_count_table(
        path=record["path"],
        geo_sample_id=record["geo_sample_id"],
        sample_id=record["sample_id"],
        chunk_rows=CHUNK_ROWS,
    )

    if sample.n_vars != EXPECTED_N_FEATURES:
        raise AssertionError(
            f"{record['path'].name}: expected {EXPECTED_N_FEATURES} "
            f"features, observed {sample.n_vars}"
        )

    if reference_features is None:
        reference_features = sample.var_names.copy()
        reference_var = sample.var.copy()
    else:
        if set(sample.var_names) != set(reference_features):
            missing = list(
                set(reference_features) - set(sample.var_names)
            )[:10]
            extra = list(
                set(sample.var_names) - set(reference_features)
            )[:10]
            raise ValueError(
                f"{record['path'].name}: feature set differs from the "
                f"first sample. Example missing={missing}; extra={extra}"
            )

        if not sample.var_names.equals(reference_features):
            sample = sample[:, reference_features].copy()

        observed_symbols = (
            sample.var.loc[reference_features, "gene_symbols"]
            .fillna("")
            .astype(str)
        )
        expected_symbols = (
            reference_var["gene_symbols"].fillna("").astype(str)
        )

        mismatched = (
            observed_symbols.to_numpy()
            != expected_symbols.to_numpy()
        )
        if mismatched.any():
            examples = reference_features[mismatched][:10].tolist()
            raise ValueError(
                f"{record['path'].name}: gene-symbol annotations differ "
                f"from the first sample for features such as {examples}"
            )

    sample_summary_rows.append(
        {
            "geo_sample_id": record["geo_sample_id"],
            "sample_id": record["sample_id"],
            "cells": int(sample.n_obs),
            "features": int(sample.n_vars),
            "total_counts": int(sample.X.sum()),
            "maximum_count": int(sample.X.max()),
            "source_file": record["path"].name,
        }
    )
    sample_objects.append(sample)

    print(sample)
    gc.collect()

sample_summary = pd.DataFrame(sample_summary_rows)
sample_summary.to_csv(
    AUDIT_DIR / "GSE141383_raw_build_sample_summary.tsv",
    sep="\t",
    index=False,
)

sample_summary

Reading GSM4202144_PJ052.counts.txt.gz ...
AnnData object with n_obs × n_vars = 1839 × 60725
    obs: 'cell_barcode', 'sample', 'sample_id', 'batch', 'geo_sample_id', 'geo_id', 'core_dataset', 'source_file'
    var: 'gene_symbols'
    uns: 'source_file', 'source_geo_accession', 'source_gene_id_header', 'source_gene_symbol_header'
Reading GSM4202145_PJ053.counts.txt.gz ...
AnnData object with n_obs × n_vars = 4459 × 60725
    obs: 'cell_barcode', 'sample', 'sample_id', 'batch', 'geo_sample_id', 'geo_id', 'core_dataset', 'source_file'
    var: 'gene_symbols'
    uns: 'source_file', 'source_geo_accession', 'source_gene_id_header', 'source_gene_symbol_header'
Reading GSM4202146_PW016-703.counts.txt.gz ...
AnnData object with n_obs × n_vars = 2811 × 60725
    obs: 'cell_barcode', 'sample', 'sample_id', 'batch', 'geo_sample_id', 'geo_id', 'core_dataset', 'source_file'
    var: 'gene_symbols'
    uns: 'source_file', 'source_geo_accession', 'source_gene_id_header', 'source_gene_symbol_header'


,geo_sample_id,sample_id,cells,features,total_counts,maximum_count,source_file
0,GSM4202144,PJ052,1839,60725,2785014,288,GSM4202144_PJ052.counts.txt.gz
1,GSM4202145,PJ053,4459,60725,5699912,269,GSM4202145_PJ053.counts.txt.gz
2,GSM4202146,PW016-703,2811,60725,3108053,250,GSM4202146_PW016-703.counts.txt.gz
3,GSM4202147,PW017-703,2831,60725,2720008,283,GSM4202147_PW017-703.counts.txt.gz
4,GSM4202148,PW032-706,4747,60725,10139456,240,GSM4202148_PW032-706.counts.txt.gz
5,GSM4202149,PW032-710,992,60725,2352968,902,GSM4202149_PW032-710.counts.txt.gz
6,GSM5975476,PW035-710,2205,60725,1585339,254,GSM5975476_PW035-710.counts.txt.gz
7,GSM5975477,PDC001,7738,60725,16859188,804,GSM5975477_PDC001.counts.txt.gz
8,GSM5975478,PW039-705,1793,60725,2808266,585,GSM5975478_PW039-705.counts.txt.gz


In [6]:
adata_merged = ad.concat(
    sample_objects,
    axis="obs",
    join="inner",
    merge="same",
    index_unique=None,
    pairwise=False,
)
adata_merged.var = reference_var.loc[adata_merged.var_names].copy()

if not adata_merged.obs_names.is_unique:
    raise AssertionError("Merged cell identifiers are not unique.")
if adata_merged.shape != (
    EXPECTED_TOTAL_CELLS,
    EXPECTED_N_FEATURES,
):
    raise AssertionError(
        "The raw-count build did not reproduce the historical DS2 shape. "
        f"Expected {(EXPECTED_TOTAL_CELLS, EXPECTED_N_FEATURES)}, "
        f"observed {adata_merged.shape}. Review the generated sample "
        "summary before continuing."
    )
if not sp.issparse(adata_merged.X):
    raise TypeError("The merged count matrix is unexpectedly dense.")

values = adata_merged.X.data
if not np.isfinite(values).all():
    raise AssertionError("Non-finite counts detected after merging.")
if np.any(values < 0):
    raise AssertionError("Negative counts detected after merging.")
if not np.equal(values, np.floor(values)).all():
    raise AssertionError("Non-integer counts detected after merging.")

adata_merged.uns["dataset_id"] = "DS2_GSE141383"
adata_merged.uns["geo_accession"] = "GSE141383"
adata_merged.uns["build_timestamp"] = datetime.now().isoformat(
    timespec="seconds"
)
adata_merged.uns["source_files"] = [
    record["path"].name for record in input_records
]
adata_merged.uns["input_format"] = (
    "GEO supplementary gzip-compressed tab-delimited raw-count tables"
)

build_summary = {
    "dataset_id": "DS2_GSE141383",
    "geo_accession": "GSE141383",
    "n_samples": int(adata_merged.obs["sample"].nunique()),
    "n_cells": int(adata_merged.n_obs),
    "n_features": int(adata_merged.n_vars),
    "matrix_format": type(adata_merged.X).__name__,
    "matrix_dtype": str(adata_merged.X.dtype),
    "total_counts": int(adata_merged.X.sum()),
    "maximum_count": int(adata_merged.X.max()),
    "output_h5ad": str(OUTPUT_H5AD),
}

with open(
    AUDIT_DIR / "GSE141383_raw_build_summary.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(build_summary, handle, indent=2)

print(adata_merged)
print(adata_merged.obs["sample"].value_counts())
print(build_summary)

AnnData object with n_obs × n_vars = 29415 × 60725
    obs: 'cell_barcode', 'sample', 'sample_id', 'batch', 'geo_sample_id', 'geo_id', 'core_dataset', 'source_file'
    var: 'gene_symbols'
    uns: 'dataset_id', 'geo_accession', 'build_timestamp', 'source_files', 'input_format'
sample
PDC001       7738
PW032-706    4747
PJ053        4459
PW017-703    2831
PW016-703    2811
PW035-710    2205
PJ052        1839
PW039-705    1793
PW032-710     992
Name: count, dtype: int64
{'dataset_id': 'DS2_GSE141383', 'geo_accession': 'GSE141383', 'n_samples': 9, 'n_cells': 29415, 'n_features': 60725, 'matrix_format': 'csr_matrix', 'matrix_dtype': 'int32', 'total_counts': 48058204, 'maximum_count': 902, 'output_h5ad': 'C:\\Users\\AmirSilco\\Documents\\GitHub\\glioma-cnv-single-cell-atlas\\data\\interim\\DS2_GSE141383\\GSE141383_merged.h5ad'}


In [8]:
# Save the merged DS2 AnnData object atomically, then reopen it
# in backed mode and validate the saved file.

OUTPUT_H5AD.parent.mkdir(parents=True, exist_ok=True)

temporary_output = OUTPUT_H5AD.with_name(
    f"{OUTPUT_H5AD.stem}.temporary{OUTPUT_H5AD.suffix}"
)

if temporary_output.exists():
    temporary_output.unlink()

adata_merged.write_h5ad(
    temporary_output,
    compression="gzip",
)

temporary_output.replace(OUTPUT_H5AD)


# AnnData does not implement the context-manager protocol in the
# installed version, so the backed file must be closed explicitly.
check = sc.read_h5ad(
    OUTPUT_H5AD,
    backed="r",
)

try:
    expected_shape = (
        EXPECTED_TOTAL_CELLS,
        EXPECTED_N_FEATURES,
    )

    if check.shape != expected_shape:
        raise AssertionError(
            "Saved-file validation failed: "
            f"expected {expected_shape}, observed {check.shape}"
        )

    if check.obs_names.has_duplicates:
        raise AssertionError(
            "Saved output contains duplicated cell identifiers."
        )

    if check.var_names.has_duplicates:
        raise AssertionError(
            "Saved output contains duplicated feature identifiers."
        )

finally:
    check.file.close()


if not OUTPUT_H5AD.exists():
    raise FileNotFoundError(
        f"Expected output file was not created: {OUTPUT_H5AD}"
    )

print("Saved and validated:", OUTPUT_H5AD)
print(
    "Validated shape:",
    (EXPECTED_TOTAL_CELLS, EXPECTED_N_FEATURES),
)

Saved and validated: C:\Users\AmirSilco\Documents\GitHub\glioma-cnv-single-cell-atlas\data\interim\DS2_GSE141383\GSE141383_merged.h5ad
Validated shape: (29415, 60725)
